# RQ6 - Robustness and Generalization

**Research Question:** How robust is the best-performing supervised learning model under different train-test split ratios and cross-validation settings?

This notebook loads the raw dataset and saves the actual result table as CSV and the actual figure as PDF.

In [ ]:

# Predictive Maintenance ML Assignment - Common Setup
# This notebook starts from the raw dataset file and generates the actual table/figure for this research question.

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

# ========== USER SETTINGS ==========
# Works with CSV and Excel. Update this path if you run the notebook on Kaggle or your own machine.
DATA_PATH = "../data/predictive_maintenance_cleaned_New.csv"

# If the notebook is run from another working directory, also try /mnt/data.
if not os.path.exists(DATA_PATH):
    alt_path = "/mnt/data/predictive_maintenance_cleaned_New.csv"
    if os.path.exists(alt_path):
        DATA_PATH = alt_path

RESULTS_TABLE_DIR = "../results/tables"
RESULTS_FIGURE_DIR = "../results/figures"
os.makedirs(RESULTS_TABLE_DIR, exist_ok=True)
os.makedirs(RESULTS_FIGURE_DIR, exist_ok=True)

TARGET = "failure_within_24h"
LEAKAGE_COLUMNS = ["rul_hours", "failure_type", "estimated_repair_cost"]
RANDOM_STATE = 42

def load_dataset(path=DATA_PATH):
    """Load CSV or Excel dataset."""
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    return pd.read_csv(path)

def add_time_features(df):
    """Create time-based features from timestamp if available."""
    df = df.copy()
    if "timestamp" in df.columns:
        ts = pd.to_datetime(df["timestamp"], errors="coerce")
        df["hour"] = ts.dt.hour
        df["day_of_week"] = ts.dt.dayofweek
        df["month"] = ts.dt.month
        df["is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)
        df = df.drop(columns=["timestamp"])
    return df

def prepare_xy(df, use_leakage=False, include_machine_id=True, include_time_features=True):
    """Prepare feature matrix X and target y."""
    df = df.copy()
    if include_time_features:
        df = add_time_features(df)
    else:
        if "timestamp" in df.columns:
            df = df.drop(columns=["timestamp"])

    if TARGET not in df.columns:
        raise ValueError(f"Target column '{TARGET}' not found in dataset.")

    drop_cols = [TARGET]
    if not use_leakage:
        drop_cols += [c for c in LEAKAGE_COLUMNS if c in df.columns]
    if not include_machine_id and "machine_id" in df.columns:
        drop_cols.append("machine_id")

    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df[TARGET].astype(int)
    return X, y

def split_data(X, y, test_size=0.2):
    return train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=RANDOM_STATE
    )

def get_feature_types(X):
    categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numerical_features = [c for c in X.columns if c not in categorical_features]
    return numerical_features, categorical_features

def make_preprocessor(X, scale_numeric=True):
    numerical_features, categorical_features = get_feature_types(X)

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(steps=numeric_steps)

    try:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot)
    ])

    transformers = []
    if numerical_features:
        transformers.append(("num", numeric_transformer, numerical_features))
    if categorical_features:
        transformers.append(("cat", categorical_transformer, categorical_features))

    return ColumnTransformer(transformers=transformers, remainder="drop")

def get_model(name):
    """Return a classification model by name."""
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
    if name == "Decision Tree":
        return DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=RANDOM_STATE)
    if name == "k-NN":
        return KNeighborsClassifier(n_neighbors=7)
    if name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=250, max_depth=None, min_samples_leaf=2,
            class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
        )
    if name == "SVM":
        return SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=RANDOM_STATE)
    if name == "XGBoost":
        if XGBOOST_AVAILABLE:
            return XGBClassifier(
                n_estimators=250, learning_rate=0.05, max_depth=4,
                subsample=0.9, colsample_bytree=0.9,
                eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1
            )
        return GradientBoostingClassifier(random_state=RANDOM_STATE)
    if name == "Gradient Boosting":
        return GradientBoostingClassifier(random_state=RANDOM_STATE)
    raise ValueError(f"Unknown model name: {name}")

def build_pipeline(model_name, X, scale_numeric=True):
    return Pipeline(steps=[
        ("preprocessor", make_preprocessor(X, scale_numeric=scale_numeric)),
        ("model", get_model(model_name))
    ])

def predict_scores(model, X_test):
    """Return probability score for positive class where possible."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_test)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
        # Min-max scale decision scores to [0, 1] for AUC compatibility only
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
    return None

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_score = predict_scores(model, X_test)
    auc = roc_auc_score(y_test, y_score) if y_score is not None and len(np.unique(y_test)) > 1 else np.nan
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "AUC": auc
    }

def format_metric_table(df, metric_cols=None):
    out = df.copy()
    if metric_cols is None:
        metric_cols = [c for c in ["Accuracy", "Precision", "Recall", "F1-score", "AUC"] if c in out.columns]
    for col in metric_cols:
        out[col] = out[col].astype(float).round(4)
    return out

def save_table(df, filename):
    path = os.path.join(RESULTS_TABLE_DIR, filename)
    df.to_csv(path, index=False)
    print(f"Saved table: {path}")
    return path

def save_figure(filename):
    path = os.path.join(RESULTS_FIGURE_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, format="pdf", bbox_inches="tight")
    print(f"Saved figure: {path}")
    plt.show()
    return path

def add_dummy_note():
    plt.figtext(0.5, -0.02, "Actual results generated from the dataset.", ha="center", fontsize=9, style="italic")


In [ ]:

# RQ6: Robustness and Generalization

df = load_dataset()
X, y = prepare_xy(df, use_leakage=False, include_machine_id=True, include_time_features=True)

selected_model_name = "Random Forest"
scenarios = [
    ("70/30 Split", 0.30),
    ("80/20 Split", 0.20),
    ("90/10 Split", 0.10),
]

rows = []

for scenario_name, test_size in scenarios:
    X_train, X_test, y_train, y_test = split_data(X, y, test_size=test_size)
    scale = selected_model_name in ["Logistic Regression", "k-NN", "SVM"]
    model = build_pipeline(selected_model_name, X_train, scale_numeric=scale)
    model.fit(X_train, y_train)
    metrics = evaluate_model(model, X_test, y_test)
    metrics["Scenario"] = scenario_name
    metrics["Std. Dev."] = np.nan
    rows.append(metrics)

# 5-fold cross-validation
scale = selected_model_name in ["Logistic Regression", "k-NN", "SVM"]
cv_model = build_pipeline(selected_model_name, X, scale_numeric=scale)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = cross_validate(cv_model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
cv_row = {
    "Scenario": "5-Fold CV",
    "Accuracy": np.mean(cv_results["test_accuracy"]),
    "Precision": np.mean(cv_results["test_precision"]),
    "Recall": np.mean(cv_results["test_recall"]),
    "F1-score": np.mean(cv_results["test_f1"]),
    "AUC": np.mean(cv_results["test_roc_auc"]),
    "Std. Dev.": np.std(cv_results["test_f1"])
}
rows.append(cv_row)

table = pd.DataFrame(rows)[["Scenario", "Accuracy", "Precision", "Recall", "F1-score", "AUC", "Std. Dev."]]
table = format_metric_table(table, metric_cols=["Accuracy", "Precision", "Recall", "F1-score", "AUC", "Std. Dev."])
display(table)
save_table(table, "RQ6_robustness_analysis.csv")

# Figure 6: line chart with F1-score and AUC
plt.figure(figsize=(10, 6))
x = np.arange(len(table))
std_vals = table["Std. Dev."].fillna(0.01).values

plt.errorbar(x, table["F1-score"], yerr=std_vals, marker="o", linewidth=2, capsize=4, label="F1-score (mean ± std)")
plt.errorbar(x, table["AUC"], yerr=std_vals, marker="s", linewidth=2, capsize=4, label="AUC (mean ± std)")

for i, row in table.iterrows():
    plt.text(i, row["F1-score"] + 0.025, f"{row['F1-score']:.2f}", ha="center", fontsize=8)
    plt.text(i, row["AUC"] + 0.025, f"{row['AUC']:.2f}", ha="center", fontsize=8)

plt.xticks(x, table["Scenario"])
plt.ylim(0, 1.0)
plt.title("Figure 6. Robustness of the Selected Model Under Different Validation Settings", fontsize=14, weight="bold")
plt.xlabel("Validation Setting")
plt.ylabel("Score")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.legend(loc="lower left")
plt.annotate(
    "Performance stability can be assessed\nfrom changes across validation settings.",
    xy=(len(table)-1, table["F1-score"].iloc[-1]),
    xytext=(1.8, 0.25),
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.8)
)
plt.figtext(0.5, -0.02, f"Actual results generated from the dataset; selected model = {selected_model_name}.", ha="center", fontsize=9, style="italic")
save_figure("Figure_6_robustness_validation_settings.pdf")
